
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10ea452b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ea45fd0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10ea452b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ea45fd0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie

In [4]:
model.invoke("Provide details about the moview Inception")

AIMessage(content="**Inception** (2010) – a mind‑bending science‑fiction thriller directed by Christopher\u202fNolan\n\n| Category | Details |\n|----------|---------|\n| **Release date** | **July\u202f16,\u202f2010** (USA) |\n| **Runtime** | 2\u202fh\u202f28\u202fmin |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | ≈\u202f$160\u202fmillion |\n| **Box‑office** | ≈\u202f$828\u202fmillion worldwide (≈\u202f$536\u202fmillion domestic) |\n| **Production companies** | Warner Bros., Syncopy, Legendary Pictures |\n| **Screenwriter** | Christopher\u202fNolan (co‑written with Jonathan Nolan) |\n| **Director** | Christopher\u202fNolan |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Wally\u202fPfister |\n| **Editing** | Lee\u202fSmith |\n\n### Premise\nA skilled thief, **Dom Cobb** (Leonardo\u202fDiCaprio), steals secrets by infiltrating the subconscious during dreams. He is offered a chance to erase his criminal record: perform an *inception*—plant an ide

In [5]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### MEssage output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie") ##Field(...) means that this field is required and must be provided when creating an instance of the Movie model.
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie Inception. The user just wants details. We can use the function to fetch details. The function requires director, rating, title, year. We can provide these. But the function returns "any". We need to call it. The function name is "Movie". We must call the function with arguments: director: Christopher Nolan, rating: maybe 8.8, title: Inception, year: 2010. Then we can display the result. The user didn\'t specify format. We\'ll just provide details. We\'ll call the function.', 'tool_calls': [{'id': 'fc_bfc28797-8b21-4253-91a2-c14fce0fcf9b', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 164, 'total_tokens': 321, 'completion_time': 0.195187547, 'completion_tokens_details': {'reasoning_tokens': 118}, 

## What does include_raw=True do?
means you want both:

1. the parsed structured result
2. the raw model response

In [14]:
response['parsed']  # Access the parsed output as a Movie instance

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Nested Structure

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation. it gives output in dictionary form.

In [18]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"] ## meaning : title should be a required string, and its description is "The title of the movie".
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie The Avengers (2008)")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2008}

In [22]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Please provide the details of the movie The Avengers (2008)")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark/Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers/Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff/Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner/Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton/Hawkeye'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2008}

In [23]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='56f49aee-0209-4fef-b648-8673beafcd1c'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object conforming to ContactInfo schema. Fields: name, email, phone. All required. Provide compact JSON. Ensure valid JSON. Output only JSON object.'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 238, 'total_tokens': 312, 'completion_time': 0.12024573, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.011765657, 'prompt_tokens_details': None, 'queue_time': 0.3522248, 'total_time': 0.132011387}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11a

In [25]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [26]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='dcef6e2b-6e8d-4d2f-80da-cf9d185dc0e2'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object that matches schema. Fields: name, email, phone. All required. Must be compact JSON. Provide only final JSON object. Ensure valid JSON. Let\'s produce: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 212, 'total_tokens': 311, 'completion_time': 0.145553219, 'completion_tokens_details': {'reasoning_tokens': 63}, 'prompt_time': 0.014596659, 'prompt_tokens_details': None, 'queue_time': 0.212952379, 'total_time': 0.160149878}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'fi

In [27]:
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [28]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='58b9cb3e-f5d3-411e-9caa-34ab82dbf2fd'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON matching the ContactInfo schema. Must include all required fields: name, email, phone. Provide compact JSON. Let\'s construct:\n\n{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}\n\nCheck schema: title etc not needed in output. Just the object. Ensure compact.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 212, 'total_tokens': 315, 'completion_time': 0.122524948, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.013139437, 'prompt_tokens_details': None, 'queue_time': 0.045329712, 'total_time': 0.135664385}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerpri

In [29]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### @dataclass gives you a simple Python class that describes the structure you want the agent to return. id does not validate structured data as Pydantic BaseModel does.
### @dataclass is python built in functionality whereas Pydantic is a external library.